In [ ]:
# !pip install -U langchain langchain-core langchain-google-genai requests

In [ ]:
# import os
# from getpass import getpass

# os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

In [2]:
import os
import requests

#from google.colab import    

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


from dotenv import load_dotenv

load_dotenv("backend/.env")

api_key = os.getenv("GOOGLE_API_KEY")


# ============================================================
# 2. TOOL 1: Geocoding
# ============================================================

@tool
def geocode_city_name(city_name: str) -> str:
    """Find latitude and longitude for a city."""

    print("\n🔧 TOOL CALLED: geocode_city_name")
    print(f"   Input: city_name = {city_name}")

    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": city_name,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    results = data.get("results", [])

    if not results:
        observation = f"Could not find city: {city_name}"

        print(f"   👁️ OBSERVATION: {observation}")

        return observation

    result = results[0]

    observation = (
        f"City: {result.get('name')}\n"
        f"Country: {result.get('country')}\n"
        f"Latitude: {result.get('latitude')}\n"
        f"Longitude: {result.get('longitude')}"
    )

    print("   👁️ OBSERVATION:")
    print(f"   {observation}")

    return observation


# ============================================================
# 3. TOOL 2: Weather
# ============================================================

@tool
def get_current_weather(
    latitude: float,
    longitude: float
) -> str:
    """Get current weather for latitude and longitude."""

    print("\n🔧 TOOL CALLED: get_current_weather")
    print(f"   Input: latitude = {latitude}")
    print(f"   Input: longitude = {longitude}")

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": [
            "temperature_2m",
            "relative_humidity_2m",
            "apparent_temperature",
            "precipitation",
            "weather_code",
            "wind_speed_10m"
        ],
        "timezone": "auto"
    }

    response = requests.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    current = data.get("current")

    if not current:
        observation = "No current weather data found."

        print(f"   👁️ OBSERVATION: {observation}")

        return observation

    observation = str(current)

    print("   👁️ OBSERVATION:")
    print(f"   {observation}")

    return observation


# ============================================================
# 4. TOOLS
# ============================================================

tools = [
    geocode_city_name,
    get_current_weather,
]


# ============================================================
# 5. GEMINI
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0
)


# ============================================================
# 6. AGENT
# ============================================================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a weather assistant.

When the user asks about a city:

1. First use geocode_city_name.
2. Get latitude and longitude.
3. Then use get_current_weather.
4. Finally explain the weather to the user.

You must use the tools instead of guessing.
"""
)


# ============================================================
# 7. USER QUESTION
# ============================================================

query = "What is the weather like in Mumbai right now?"


print("\n")
print("=" * 70)
print("USER")
print("=" * 70)

print(query)


# ============================================================
# 8. RUN AGENT
# ============================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ]
    }
)


# ============================================================
# 9. INSPECT EVERY MESSAGE
# ============================================================

print("\n")
print("=" * 70)
print("AGENT TRACE")
print("=" * 70)


for i, message in enumerate(result["messages"]):

    print("\n")
    print("-" * 70)

    print(f"MESSAGE #{i}")
    print("-" * 70)

    print("TYPE:")
    print(type(message).__name__)

    print("\nCONTENT:")
    print(message.content)

    # --------------------------------------------------------
    # Tool calls made by the LLM
    # --------------------------------------------------------

    if hasattr(message, "tool_calls") and message.tool_calls:

        print("\n📞 TOOL CALLS:")

        for tool_call in message.tool_calls:

            print(f"  Tool: {tool_call['name']}")
            print(f"  Arguments: {tool_call['args']}")
            print(f"  Tool Call ID: {tool_call['id']}")

    # --------------------------------------------------------
    # Tool result / observation
    # --------------------------------------------------------

    if type(message).__name__ == "ToolMessage":

        print("\n👁️ OBSERVATION / TOOL RESULT:")

        print(message.content)


# ============================================================
# 10. FINAL ANSWER
# ============================================================

print("\n")
print("=" * 70)
print("FINAL ANSWER")
print("=" * 70)

print(result["messages"][-1].content)



USER
What is the weather like in Mumbai right now?

🔧 TOOL CALLED: geocode_city_name
   Input: city_name = Mumbai
   👁️ OBSERVATION:
   City: Mumbai
Country: India
Latitude: 19.07283
Longitude: 72.88261

🔧 TOOL CALLED: get_current_weather
   Input: latitude = 19.07283
   Input: longitude = 72.88261
   👁️ OBSERVATION:
   {'time': '2026-08-31T14:30', 'interval': 900, 'temperature_2m': 28.9, 'relative_humidity_2m': 74, 'apparent_temperature': 33.1, 'precipitation': 0.0, 'weather_code': 1, 'wind_speed_10m': 17.8}


AGENT TRACE


----------------------------------------------------------------------
MESSAGE #0
----------------------------------------------------------------------
TYPE:
HumanMessage

CONTENT:
What is the weather like in Mumbai right now?


----------------------------------------------------------------------
MESSAGE #1
----------------------------------------------------------------------
TYPE:
AIMessage

CONTENT:
[]

📞 TOOL CALLS:
  Tool: geocode_city_name
  Arguments: {